# Agder – dump 10 m feature rasters (region-specific)

**Not part of the main national workflow.** Companion to
[01_agder_substrat_10m.ipynb](./01_agder_substrat_10m.ipynb): builds the model
input features for **Agder** at **10 m** and writes each of the 8
`DEPTH_NAMES` features as its own GeoTIFF via
`subkart.utils.save_feature_rasters`.

The key output is `agder_10m_dem_depth.tif` — the DEM depth feature **after
missing values are filled** (NaN DEM cells filled from the interpolated sea-map
depth in `features.build`).


## Parameters & inputs

In [1]:
from pathlib import Path

import geopandas as gpd
import geoutils as gu
import numpy as np
import shapely

import mnk
import mnk.substrat as subkart

REGION = "agder"
FYLKER = ["Agder"]
RES = 10
nodata = 255
crs = "EPSG:25833"

out_dir = Path("features")
out_dir.mkdir(exist_ok=True)
prefix = f"{REGION}_{RES}m"
print("Writing features to:", out_dir.resolve(), "prefix:", prefix)

Writing features to: /home/kim/work/marint-naturkart-nivaR/notebooks/01_substrat/regions/features prefix: agder_10m


In [2]:
# National rasters (cropped/reprojected to Agder below)
bolge = mnk.sources.bolge_exposure()
dem_norge = mnk.sources.dem_data()

# Agder-only sea map + depth points
gdf_sea_map = mnk.sources.sea_map_basisdata(fylker=FYLKER)
gdf_sea_map = subkart.features.depth_preprocess(gdf_sea_map)
transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map, res=RES)
print(f"Output raster shape: {out_shape}  (res={RES} m)")

dem = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, crs)

# Crop bolge to the Agder window *before* reprojecting. geoutils' reproject reads
# the full source raster into memory, so reprojecting the national raster directly
# would load all of Norway. crop() reads only the windowed subset from disk.
# A buffer avoids edge artefacts from bilinear resampling at the crop boundary.
BOLGE_BUFFER = 1000  # metres
bolge_bbox = gu.Vector(
    gpd.GeoDataFrame(
        geometry=[shapely.box(
            bounds[0] - BOLGE_BUFFER, bounds[1] - BOLGE_BUFFER,
            bounds[2] + BOLGE_BUFFER, bounds[3] + BOLGE_BUFFER,
        )],
        crs=crs,
    )
)
bolge_agder = bolge.crop(bolge_bbox).reproject(
    crs=crs, res=RES,
    bounds=dict(left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3]),
)

gdf_points = mnk.sources.depth_point_data(fylker=FYLKER)
print(f"Depth points: {len(gdf_points):,}")

Output raster shape: (10266, 21222)  (res=10 m)
Depth points: 160,001


## Build features and dump the 8 DEPTH_NAMES rasters

In [3]:
X, valid, out_shape, transform = subkart.features.build(
    dem, gdf_sea_map, bolge_agder, valid_mask=None, res=RES, dtype=np.float32,
    gdf_points=gdf_points,
)
print("Feature columns:", subkart.features.NAMES)
print("X shape:", X.shape)

# Free the large build inputs now that features are materialised in X; at 10 m each
# full-grid raster is ~0.9 GB, so releasing them before writing cuts peak memory.
dem_nodata = dem.nodata
del dem, bolge, bolge_agder, gdf_sea_map, gdf_points
import gc; gc.collect()

Computing interpolated depth raster...
Preparing sea_avg_depth...
Preparing sea_avg_slope...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_compactness...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_convexity...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing wave exposure...
Stacking feature arrays...
Feature columns: ['dem_depth', 'dem_slope', 'dem_curvature', 'is_dem', 'sea_avg_depth', 'sea_avg_slope', 'sea_compactness', 'sea_convexity', 'wave_exposure']
X shape: (16123727, 9)


0

In [4]:
# save_feature_rasters writes one GeoTIFF per DEPTH_NAMES feature (8 bands),
# including the NaN-filled dem_depth. wave_exposure is intentionally not dumped.
subkart.utils.save_feature_rasters(
    out_dir, prefix, X, valid, transform, out_shape, dem_nodata, crs=crs,
)
for name in subkart.features.DEPTH_NAMES:
    print("Saved:", out_dir / f"{prefix}_{name}.tif")

# Release the stacked feature matrix and valid mask once written.
del X, valid
gc.collect()

Saved: features/agder_10m_dem_depth.tif
Saved: features/agder_10m_dem_slope.tif
Saved: features/agder_10m_dem_curvature.tif
Saved: features/agder_10m_is_dem.tif
Saved: features/agder_10m_sea_avg_depth.tif
Saved: features/agder_10m_sea_avg_slope.tif
Saved: features/agder_10m_sea_compactness.tif
Saved: features/agder_10m_sea_convexity.tif


0